# Model Training Pipeline (Based on SWIFT Framework)

> **Note**: This notebook documents the training procedure.
>
> The final model weights are publicly available at `https://huggingface.co/BGI-LHH/VIPER`.

## Example: VIPER Model
- **Base model**: Qwen3-1.7B
- **Training data**: ~40,000 examples in JSONL format (Trainset-VIPER.jsonl)
- **Average sequence length**: ~2,000 tokens per example
- **Hardware**: 2× NVIDIA A100 (40GB) GPUs
- **Peak GPU memory**: ~28GB per GPU
- **Training time**: < 8 hours

## Assumptions:
- Training data (`train_jsonl`) is already prepared in JSONL format.
- Python environment with `swift` and all dependencies is properly set up.
- Target GPUs are available (`CUDA_VISIBLE_DEVICES=0,1`).


In [1]:
# Verify that swift is installed and show its version
import swift
print(f"SWIFT version: {swift.__version__}")
print(f"SWIFT path: {swift.__file__}")

SWIFT version: 3.6.0
SWIFT path: /home/.conda/envs/SwiftTest/lib/python3.12/site-packages/swift/__init__.py


## Training Configuration

We use full parameter fine-tuning with the following key settings:
- Context length: **32768 tokens**
- Per-device batch size: **1** (with gradient accumulation)
- Precision: **bfloat16**
- Attention: **FlashAttention v2 (`flash_attn`)**
- Distributed training: **DDP + DeepSpeed ZeRO Stage 2**
- Optimizer: AdamW with weight decay (`0.1`), learning rate `1e-5`, and linear warmup

> Full CLI options: https://github.com/modelscope/ms-swift
>
> Since model weights are already released, **running this training is optional** for reviewers.

In [2]:
# Display the training script content for transparency
training_command = r'''
export nproc_per_node=2
export NPROC_PER_NODE=$nproc_per_node
export CUDA_VISIBLE_DEVICES=2,3
export MASTER_PORT=29887

# Launch full fine-tuning via SWIFT
swift sft \
    --model {model_dir} \
    --model_type {model_type} \
    --output_dir {output_dir} \
    --check_model false \
    --ddp_backend nccl \
    --train_type full \
    --torch_dtype bfloat16 \
    --dataset {train_jsonl} \
    --num_train_epochs 3 \
    --max_length 32768 \
    --gradient_checkpointing true \
    --per_device_train_batch_size 1 \
    --weight_decay 0.1 \
    --learning_rate 1e-5 \
    --gradient_accumulation_steps $(expr 12 / $nproc_per_node) \
    --max_grad_norm 0.5 \
    --warmup_ratio 0.03 \
    --eval_steps 30 \
    --save_steps 30 \
    --logging_steps 10 \
    --attn_impl flash_attn \
    --deepspeed zero2 \
    --save_only_model false \
    --save_total_limit 2 \
    --padding_free true
'''

print(training_command)


export nproc_per_node=2
export NPROC_PER_NODE=$nproc_per_node
export CUDA_VISIBLE_DEVICES=2,3
export MASTER_PORT=29887

# Launch full fine-tuning via SWIFT
swift sft \
    --model {model_dir} \
    --model_type {model_type} \
    --output_dir {output_dir} \
    --check_model false \
    --ddp_backend nccl \
    --train_type full \
    --torch_dtype bfloat16 \
    --dataset {train_jsonl} \
    --num_train_epochs 3 \
    --max_length 32768 \
    --gradient_checkpointing true \
    --per_device_train_batch_size 1 \
    --weight_decay 0.1 \
    --learning_rate 1e-5 \
    --gradient_accumulation_steps $(expr 12 / $nproc_per_node) \
    --max_grad_norm 0.5 \
    --warmup_ratio 0.03 \
    --eval_steps 30 \
    --save_steps 30 \
    --logging_steps 10 \
    --attn_impl flash_attn \
    --deepspeed zero2 \
    --save_only_model false \
    --save_total_limit 2 \
    --padding_free true



In [3]:
# Optional: Uncomment and fill in actual paths to run training
# Replace placeholders like {model_dir}, {train_jsonl}, etc.

# Example:
# MODEL_DIR="/path/to/base/model"
# TRAIN_DATA="/data/train.jsonl"
# OUTPUT_DIR="./output"

# !CUDA_VISIBLE_DEVICES=2,3 swift sft \
#     --model $MODEL_DIR \
#     --model_type qwen3 \
#     --output_dir $OUTPUT_DIR \
#     --dataset $TRAIN_DATA \
#     --train_type full \
#     --max_length 32768 \
#     --per_device_train_batch_size 1 \
#     --gradient_accumulation_steps 6 \
#     --torch_dtype bfloat16 \
#     --attn_impl flash_attn \
#     --deepspeed zero2 \
#     --padding_free true \
#     --num_train_epochs 3 \
#     --learning_rate 1e-5 \
#     --logging_steps 10 \
#     --save_steps 30